# Phát hiện viêm phổi trên ảnh X-quang ngực

**Baseline ResNet18 — báo cáo kết quả**

Dữ liệu: [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia) —
5.856 ảnh X-quang ngực của trẻ 1–5 tuổi, hai lớp `NORMAL` và `PNEUMONIA`.

---

## Tóm tắt

| Hạng mục | Lựa chọn |
|---|---|
| Kiến trúc | ResNet18 pretrained ImageNet, fine-tune toàn bộ |
| Chia dữ liệu | Cross-validation 5 fold **theo bệnh nhân**; tập test gốc giữ nguyên làm holdout |
| Mất cân bằng lớp | Cross-entropy có trọng số lớp (train lệch 2,89:1) |
| Chọn checkpoint | Theo val F1, khôi phục trước khi chấm test |
| Điểm làm việc | Dò ngưỡng trên validation; báo cáo cả ngưỡng mặc định lẫn ngưỡng dò |
| Diễn giải | Grad-CAM kèm phép đo định lượng vùng nhiệt |

Kết quả ở phần 3, diễn giải ở phần 4, hạn chế ở phần 5.

---

### Chuẩn bị môi trường

1. **+ Add Data** → thêm `chest-xray-pneumonia` (paultimothymooney)
2. **Settings → Accelerator** → **GPU T4**
3. **Settings → Internet** → **On** (để tải ImageNet weights)

Thời gian chạy: khoảng 40 phút với `N_FOLDS = 5`; khoảng 10 phút với `N_FOLDS = 1`.

## Cấu hình

In [ ]:
SEED          = 42
IMG_SIZE      = 224
BATCH_SIZE    = 32
EPOCHS        = 15
LR            = 1e-4
WEIGHT_DECAY  = 1e-5
PATIENCE      = 5
CLASS_WEIGHTS = True
NUM_WORKERS   = 2

N_FOLDS       = 5         # 1 = một holdout 15% (~10 phút); 5 = CV đầy đủ (~40 phút)
VAL_FRACTION  = 0.15      # chỉ dùng khi N_FOLDS = 1
BORDER_FRAC   = 0.15      # dùng ở phần 4.3

CLASSES  = ("NORMAL", "PNEUMONIA")     # NORMAL = 0, PNEUMONIA = 1 (lớp dương)
WORK_DIR = "/kaggle/working"
LOG_PATH = f"{WORK_DIR}/train_log.txt"

In [ ]:
import hashlib, os, random, re, time, warnings
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

warnings.filterwarnings("ignore", category=UserWarning)

assert torch.cuda.is_available(), (
    "Chưa bật GPU. Settings -> Accelerator -> GPU T4, rồi Run All lại."
)
DEVICE = torch.device("cuda")


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def log(*parts):
    """In ra màn hình, đồng thời ghi vào LOG_PATH.

    Output của Kaggle có thể mất chunk khi in nhanh; file thì không.
    """
    line = " ".join(str(part) for part in parts)
    print(line)
    os.makedirs(WORK_DIR, exist_ok=True)
    with open(LOG_PATH, "a", encoding="utf-8") as handle:
        handle.write(line + "\n")


set_seed()
log("torch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
log("log file:", LOG_PATH)

---

# 1. Dữ liệu

## 1.1. Nạp dữ liệu

Kaggle mount bộ dữ liệu này theo hai kiểu đường dẫn tuỳ cách thêm. Bên trong còn
hai thứ gây sai số nếu quét thư mục một cách ngây thơ:

- cây `chest_xray/chest_xray/` lồng bên trong, **chứa bản sao của toàn bộ ảnh**
- cây `__MACOSX/` chứa file `._*.jpeg` — không phải ảnh, nhưng có đuôi `.jpeg`
  và có đủ cấu trúc `train/NORMAL` + `train/PNEUMONIA`

Hàm dưới dò theo thư mục mốc, loại `__MACOSX`, và chọn cây nông nhất.

In [ ]:
def list_images(directory):
    """Ảnh .jpeg thật, bỏ file sidecar ._* của macOS."""
    return sorted(p for p in Path(directory).glob("*.jpeg")
                  if not p.name.startswith("._"))


def find_data_root(search_paths):
    """Thư mục chứa trực tiếp train/NORMAL và train/PNEUMONIA."""
    if isinstance(search_paths, (str, Path)):
        search_paths = [search_paths]

    candidates = []
    for base in map(Path, search_paths):
        if not base.exists():
            continue
        for train_dir in base.rglob("train"):
            if "__MACOSX" in train_dir.parts:
                continue
            if (train_dir / "NORMAL").is_dir() and (train_dir / "PNEUMONIA").is_dir():
                candidates.append(train_dir.parent)

    if not candidates:
        raise FileNotFoundError(
            f"Không tìm thấy dataset dưới {[str(p) for p in search_paths]}. "
            "Bấm '+ Add Data' và thêm 'Chest X-Ray Images (Pneumonia)'."
        )

    candidates = sorted(set(candidates), key=lambda path: len(path.parts))
    for candidate in candidates:
        n = len(list_images(candidate / "train" / "NORMAL"))
        mark = "  <- dùng" if candidate == candidates[0] else "  (bản trùng, bỏ qua)"
        print(f"  {candidate}  [{n} ảnh train/NORMAL]{mark}")
    return candidates[0]


DATA_ROOT = find_data_root(["/kaggle/input", "../chest_xray", "data/raw"])
log("\nDATA_ROOT =", DATA_ROOT)

## 1.2. Danh sách ảnh và định danh bệnh nhân

Tên file là thông tin duy nhất cho biết ảnh nào thuộc bệnh nhân nào:

| Lớp | Dạng tên |
|---|---|
| Viêm phổi | `person1_bacteria_1.jpeg`, `person1_virus_6.jpeg` |
| Bình thường | `IM-0115-0001.jpeg`, `NORMAL2-IM-1427-0001.jpeg` |

Bộ đếm `person` **không dùng chung** giữa hai phân nhóm viêm phổi: `bacteria` và
`virus` mỗi loại chạy một dãy riêng từ 1. Khoá định danh vì vậy phải gồm cả phân
nhóm — phần 1.3.4 chứng minh bằng số.

In [ ]:
PNEUMONIA_RE = re.compile(r"^person(\d+)_(bacteria|virus)_", re.IGNORECASE)
NORMAL_RE    = re.compile(r"^(?:(NORMAL\d+)-)?IM-(\d+)-", re.IGNORECASE)


def parse_group_id(filename):
    """Khoá bệnh nhân suy từ tên file. Báo lỗi nếu gặp dạng tên lạ."""
    match = PNEUMONIA_RE.match(filename)
    if match:
        return f"pneumonia:{match.group(2).lower()}:{int(match.group(1))}"
    match = NORMAL_RE.match(filename)
    if match:
        return f"normal:{(match.group(1) or 'IM').lower()}:{int(match.group(2))}"
    raise ValueError(f"Tên file lạ, không suy ra được bệnh nhân: {filename}")


def build_manifest(root):
    """Một dòng cho mỗi ảnh: đường dẫn, split gốc, nhãn, bệnh nhân."""
    rows = []
    for split in ("train", "val", "test"):
        for class_id, class_name in enumerate(CLASSES):
            directory = Path(root) / split / class_name
            if not directory.is_dir():
                continue
            for path in list_images(directory):
                rows.append({
                    "path": str(path), "filename": path.name,
                    "split_original": split, "class_name": class_name,
                    "class_id": class_id, "group_id": parse_group_id(path.name),
                })
    if not rows:
        raise FileNotFoundError(f"Không có ảnh .jpeg nào dưới {root}")

    frame = pd.DataFrame(rows)
    frame["cache_index"] = np.arange(len(frame))   # vị trí trong cache ở phần 2.2
    return frame


manifest = build_manifest(DATA_ROOT)
log(f"{len(manifest):,} ảnh | {manifest['group_id'].nunique():,} bệnh nhân")
manifest.head(3)

## 1.3. Kiểm tra chất lượng dữ liệu

In [ ]:
print("1.3.1  Số lượng theo split và lớp")
print("-" * 62)
print(f"{'split':<8}{'NORMAL':>9}{'PNEUMONIA':>12}{'tổng':>9}{'P/N':>7}")
for split in ("train", "val", "test"):
    subset = manifest[manifest["split_original"] == split]
    counts = subset["class_name"].value_counts()
    normal, pneumonia = int(counts.get("NORMAL", 0)), int(counts.get("PNEUMONIA", 0))
    ratio = pneumonia / normal if normal else float("nan")
    print(f"{split:<8}{normal:>9,}{pneumonia:>12,}{normal + pneumonia:>9,}{ratio:>7.2f}")
print(f"{'TỔNG':<8}{'':>9}{'':>12}{len(manifest):>9,}")

Tập validation gốc chỉ có **16 ảnh** — quá nhỏ để chọn checkpoint, nên phần 2.1
cắt validation mới từ pool train+val.

Tỉ lệ hai lớp cũng khác nhau giữa các split: train **2,89:1** còn test **1,67:1**.
Điểm này quyết định cách đọc kết quả ở phần 3.

In [ ]:
print("1.3.2  Ảnh trùng nội dung (SHA-256)")
print("-" * 62)
by_hash = defaultdict(list)
for path, split in zip(manifest["path"], manifest["split_original"]):
    by_hash[hashlib.sha256(Path(path).read_bytes()).hexdigest()].append(split)

duplicates = [splits for splits in by_hash.values() if len(splits) > 1]
cross_split = [splits for splits in duplicates if len(set(splits)) > 1]
print(f"tổng file             : {len(manifest):,}")
print(f"hash duy nhất         : {len(by_hash):,}")
print(f"nhóm ảnh trùng        : {len(duplicates)}")
print(f"  trong đó xuyên split: {len(cross_split)}")
print()
print("Không có ảnh y hệt nằm ở cả train lẫn test." if not cross_split
      else "CẢNH BÁO: ảnh trùng xuyên split — kết quả đánh giá bị nhiễm.")

In [ ]:
print("1.3.3  Định dạng và độ phân giải")
print("-" * 62)
modes, sizes, unreadable = Counter(), Counter(), 0
for path in manifest["path"]:
    try:
        with Image.open(path) as image:
            modes[image.mode] += 1
            sizes[image.size] += 1
    except OSError:
        unreadable += 1

print(f"file không mở được : {unreadable}")
for mode, count in modes.most_common():
    print(f"  mode {mode:<4}       : {count:,}")
print(f"số kích thước khác : {len(sizes):,}")
print()
print("Có cả ảnh RGB lẫn grayscale, nên pipeline ép về một kênh trước khi xử lý.")

In [ ]:
print("1.3.4  Định danh bệnh nhân và nguy cơ trùng giữa các split")
print("-" * 62)
naive_re = re.compile(r"^(person\d+)_", re.IGNORECASE)
naive, corrected = defaultdict(set), defaultdict(set)
for filename, split in zip(manifest["filename"], manifest["split_original"]):
    match = naive_re.match(filename)
    naive[match.group(1).lower() if match else filename].add(split)
    corrected[parse_group_id(filename)].add(split)

naive_span     = sum(1 for s in naive.values() if len(s) > 1)
corrected_span = sum(1 for s in corrected.values() if len(s) > 1)
print(f"khoá person<N>             : {len(naive):,} nhóm, {naive_span} nằm ở >1 split")
print(f"khoá (phân nhóm, person<N>) : {len(corrected):,} nhóm, {corrected_span} nằm ở >1 split")

subtype_ids = defaultdict(set)
for filename in manifest["filename"]:
    match = PNEUMONIA_RE.match(filename)
    if match:
        subtype_ids[match.group(2).lower()].add(int(match.group(1)))
print()
for subtype, ids in sorted(subtype_ids.items()):
    print(f"  {subtype:<9}: {len(ids):,} số, dải 1..{max(ids)}, "
          f"mật độ {len(ids) / max(ids):.3f}")
print(f"  số được dùng bởi CẢ HAI phân nhóm: "
      f"{len(subtype_ids['bacteria'] & subtype_ids['virus']):,}")

per_group = Counter(parse_group_id(f) for f in manifest["filename"])
multi = sum(1 for n in per_group.values() if n > 1)
print(f"\nbệnh nhân có >1 ảnh: {multi:,}/{len(per_group):,} "
      f"(nhiều nhất {max(per_group.values())} ảnh)")

Hai dãy `bacteria` và `virus` đều gần liên tục, đều bắt đầu từ 1, và có 979 số
được dùng ở cả hai. Nếu bộ đếm là toàn cục thì dãy `virus` phải tránh các số đã
thuộc `bacteria`; thực tế chúng trùng nhau liên tục. Vậy `person1_bacteria` và
`person1_virus` là hai người khác nhau.

Với khoá đúng, **không bệnh nhân nào nằm ở nhiều hơn một split gốc** — tập test
gốc dùng làm holdout được.

Nhưng 726 bệnh nhân có nhiều hơn một ảnh, cá biệt một người 30 ảnh. Cắt validation
ở mức ảnh sẽ đẩy ảnh của cùng một người sang cả hai phía, khiến mô hình được chấm
trên chính bệnh nhân nó đã học. Đó là lý do phần 2.1 chia theo bệnh nhân.

## 1.4. Phân tích khám phá

In [ ]:
C_NORMAL, C_PNEUMONIA = "#2a78d6", "#eb6834"
CLASS_COLOR = {"NORMAL": C_NORMAL, "PNEUMONIA": C_PNEUMONIA}
INK, INK_SOFT, GRID = "#0b0b0b", "#52514e", "#e5e4e0"

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": GRID, "axes.labelcolor": INK_SOFT,
    "axes.titlecolor": INK, "axes.titlesize": 12, "axes.titleweight": "600",
    "axes.titlelocation": "left", "axes.titlepad": 12,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": INK_SOFT, "ytick.color": INK_SOFT,
    "xtick.labelsize": 10, "ytick.labelsize": 10,
    "legend.frameon": False, "font.size": 10,
})


def tidy(ax, ygrid_only=True):
    ax.tick_params(length=0)
    ax.xaxis.grid(not ygrid_only)
    ax.yaxis.grid(ygrid_only)
    return ax

### Phân bố lớp theo split

In [ ]:
counts = (manifest.groupby(["split_original", "class_name"])
          .size().unstack(fill_value=0).reindex(["train", "val", "test"]))

fig, ax = plt.subplots(figsize=(9, 4.2))
y, h, gap = np.arange(len(counts)), 0.36, 0.02
for offset, cls in ((-h / 2 - gap / 2, "NORMAL"), (h / 2 + gap / 2, "PNEUMONIA")):
    bars = ax.barh(y + offset, counts[cls].values, height=h,
                   color=CLASS_COLOR[cls], label=cls)
    for rect, value in zip(bars, counts[cls].values):
        ax.text(rect.get_width() + 60, rect.get_y() + rect.get_height() / 2,
                f"{value:,}", va="center", ha="left", fontsize=9, color=INK_SOFT)

ratios = counts["PNEUMONIA"] / counts["NORMAL"].clip(lower=1)
ax.set_yticks(y)
ax.set_yticklabels([f"{s}\n{r:.2f}:1" for s, r in ratios.items()])
ax.set_xlim(0, counts.values.max() * 1.16)
ax.set_xlabel("số ảnh")
ax.set_title("Phân bố lớp theo split  ·  tỉ lệ PNEUMONIA/NORMAL dưới tên split")
ax.legend(loc="upper right", ncol=2)
tidy(ax, ygrid_only=False)
plt.tight_layout(); plt.show()

Một mô hình đoán PNEUMONIA cho mọi ảnh đã đạt **62,5%** accuracy trên test. Nên
accuracy đơn thuần không đủ để kết luận điều gì.

### Ảnh mẫu

In [ ]:
set_seed()
fig, axes = plt.subplots(2, 5, figsize=(14, 6.8), constrained_layout=True)
for row, cls in enumerate(CLASSES):
    pool = manifest[(manifest["split_original"] == "train")
                    & (manifest["class_name"] == cls)].sample(5, random_state=SEED)
    for col, (_, item) in enumerate(pool.iterrows()):
        with Image.open(item["path"]) as image:
            original = f"{image.size[0]}×{image.size[1]}"
            axes[row, col].imshow(image.convert("L").resize((256, 256)), cmap="gray")
        axes[row, col].set_title(original, fontsize=9, color=INK_SOFT)
        axes[row, col].axis("off")
    axes[row, 0].text(-0.08, 0.5, cls, transform=axes[row, 0].transAxes,
                      rotation=90, va="center", ha="center", fontsize=11,
                      fontweight="600", color=CLASS_COLOR[cls])
fig.suptitle("Ảnh mẫu từ train (đã resize vuông như đầu vào mô hình)  ·  "
             "kích thước gốc ghi trên mỗi ảnh",
             fontsize=12, fontweight="600", x=0.02, ha="left")
plt.show()

Viêm phổi biểu hiện là vùng mờ trắng ở nhu mô phổi. Đáng chú ý: hầu hết ảnh có
**ký hiệu định hướng "R"** ở góc trên — đặc trưng không mang thông tin bệnh lý
nhưng mô hình có thể bám vào. Phần 4 kiểm tra điều đó.

### Số ảnh trên mỗi bệnh nhân

In [ ]:
per_patient = manifest.groupby("group_id").size()
distribution = per_patient.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(distribution.index, distribution.values, color=C_NORMAL, width=0.72)
for rect, value in zip(bars, distribution.values):
    if value >= distribution.max() * 0.04:
        ax.text(rect.get_x() + rect.get_width() / 2, rect.get_height() * 1.06,
                f"{value:,}", ha="center", fontsize=9, color=INK_SOFT)
ax.set_yscale("log")
ax.set_xlabel("số ảnh của một bệnh nhân")
ax.set_ylabel("số bệnh nhân (thang log)")
ax.set_title(f"Số ảnh trên mỗi bệnh nhân  ·  "
             f"{(per_patient > 1).sum():,}/{len(per_patient):,} người có nhiều hơn 1 ảnh")
tidy(ax)
plt.tight_layout(); plt.show()

### Độ sáng trung bình theo lớp

In [ ]:
set_seed()
train_pool = manifest[manifest["split_original"] == "train"]
sample = pd.concat([
    train_pool[train_pool["class_name"] == cls].sample(
        min(300, (train_pool["class_name"] == cls).sum()), random_state=SEED)
    for cls in CLASSES
])

brightness = pd.DataFrame([
    {"class_name": item["class_name"],
     "mean": np.asarray(Image.open(item["path"]).convert("L").resize((64, 64)),
                        dtype=np.float32).mean()}
    for _, item in sample.iterrows()
])

fig, ax = plt.subplots(figsize=(9, 4))
bins = np.linspace(0, 255, 46)
for cls in CLASSES:
    values = brightness[brightness["class_name"] == cls]["mean"]
    ax.hist(values, bins=bins, color=CLASS_COLOR[cls], alpha=0.62,
            label=f"{cls}  (TB {values.mean():.0f})")
span = brightness["mean"]
ax.set_xlim(max(0, span.min() - 12), min(255, span.max() + 12))
ax.set_xlabel("độ sáng trung bình của ảnh  (0 = đen, 255 = trắng)")
ax.set_ylabel("số ảnh")
ax.set_title("Phân bố độ sáng theo lớp")
ax.legend(loc="upper right")
tidy(ax)
plt.tight_layout(); plt.show()

delta = abs(brightness.groupby("class_name")["mean"].mean().diff().iloc[-1])
pooled = brightness.groupby("class_name")["mean"].std().mean()
print(f"chênh lệch trung bình giữa hai lớp: {delta:.1f} mức xám")
print(f"độ lệch chuẩn trong mỗi lớp       : {pooled:.1f}")
print(f"Cohen's d = {delta / pooled:.2f}")

Hai phân bố chồng lấn gần như hoàn toàn. Mô hình **không thể** phân biệt hai lớp
chỉ bằng độ sáng tổng thể. Kiểm tra này loại trừ một dạng học tắt; các đặc trưng
cục bộ thì phải chờ phần 4.

---

# 2. Phương pháp

## 2.1. Chia dữ liệu

Tập test gốc giữ nguyên làm holdout và không tham gia bất kỳ quyết định nào.
Validation cắt từ pool train+val bằng **cross-validation 5 fold chia theo bệnh
nhân** (`StratifiedGroupKFold`): mọi ảnh của một người luôn nằm cùng một phía, và
mỗi ảnh trong pool làm validation đúng một lần.

Chạy nhiều fold cho hai thứ mà một lần chia không cho được: **độ dao động** của
kết quả, và một **ngưỡng quyết định ổn định** (phần 3.3).

In [ ]:
def label_split(manifest, train, val, test):
    out = manifest.copy()
    out["split"] = pd.NA
    out.loc[train.index, "split"] = "train"
    out.loc[val.index,   "split"] = "val"
    out.loc[test.index,  "split"] = "test"
    return out


def make_folds(manifest, n_folds=N_FOLDS, val_fraction=VAL_FRACTION, seed=SEED):
    """Danh sách manifest, mỗi phần tử là một fold đã gán cột split."""
    pool = manifest[manifest["split_original"].isin(["train", "val"])]
    test = manifest[manifest["split_original"] == "test"]
    n_splits = max(2, round(1 / val_fraction)) if n_folds == 1 else n_folds
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    folds = [label_split(manifest, pool.iloc[train_idx], pool.iloc[val_idx], test)
             for train_idx, val_idx in
             splitter.split(pool, pool["class_id"], groups=pool["group_id"])]
    return folds[:1] if n_folds == 1 else folds


def count_leaked_groups(split):
    """Số bệnh nhân xuất hiện ở nhiều hơn một split. Phải bằng 0."""
    return int((split.groupby("group_id")["split"].nunique() > 1).sum())


def split_summary(split):
    rows = []
    for name in ("train", "val", "test"):
        subset = split[split["split"] == name]
        counts = subset["class_name"].value_counts()
        normal, pneumonia = int(counts.get("NORMAL", 0)), int(counts.get("PNEUMONIA", 0))
        rows.append({"split": name, "NORMAL": normal, "PNEUMONIA": pneumonia,
                     "tổng": normal + pneumonia,
                     "bệnh nhân": subset["group_id"].nunique(),
                     "P/N": round(pneumonia / max(normal, 1), 2)})
    return pd.DataFrame(rows).set_index("split")


FOLDS = make_folds(manifest)
log(f"\n{len(FOLDS)} fold, chia theo bệnh nhân:")
for i, split in enumerate(FOLDS):
    s = split_summary(split)
    log(f"  fold {i}: train {s.loc['train','tổng']:>5,}  val {s.loc['val','tổng']:>4,}  "
        f"test {s.loc['test','tổng']:>4,}  |  bệnh nhân ở >1 split: "
        f"{count_leaked_groups(split)}")

print("\nChi tiết fold 0:")
display(split_summary(FOLDS[0]))

## 2.2. Tiền xử lý và augmentation

Ảnh gốc trung bình khoảng 1,3 megapixel với kích thước rất khác nhau. Toàn bộ được
giải mã **một lần** về mảng `uint8` 224×224 giữ trong RAM (~294 MB). Nếu giải mã
lại mỗi epoch thì CPU thành nút thắt và GPU phải chờ.

Augmentation chỉ áp cho train. Validation và test dùng pipeline tất định — nếu
không, chỉ số đánh giá sẽ dao động theo nhiễu ngẫu nhiên.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Không có bước Resize: cache bên dưới đã ở đúng IMG_SIZE.
train_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def build_image_cache(manifest, size=IMG_SIZE):
    cache = np.zeros((len(manifest), size, size), dtype=np.uint8)
    for position, path in enumerate(manifest["path"]):
        with Image.open(path) as image:
            cache[position] = np.asarray(
                image.convert("L").resize((size, size), Image.BILINEAR))
        if (position + 1) % 1500 == 0:
            print(f"  {position + 1:,}/{len(manifest):,}")
    return cache


_started = time.time()
IMAGE_CACHE = build_image_cache(manifest)
log(f"cache: {IMAGE_CACHE.nbytes / 1e6:.0f} MB cho {len(manifest):,} ảnh "
    f"trong {time.time() - _started:.0f}s")


class XRayDataset(Dataset):
    """Phục vụ ảnh đã giải mã sẵn từ cache dùng chung."""

    def __init__(self, rows, transform):
        self.rows, self.transform = rows, transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        cache_index, label = self.rows[index]
        return self.transform(Image.fromarray(IMAGE_CACHE[cache_index])), label


def make_loaders(split, batch_size=BATCH_SIZE):
    loaders = {}
    for name, transform in (("train", train_tf), ("val", eval_tf), ("test", eval_tf)):
        subset = split[split["split"] == name]
        loaders[name] = DataLoader(
            XRayDataset(list(zip(subset["cache_index"], subset["class_id"])), transform),
            batch_size=batch_size, shuffle=(name == "train"),
            num_workers=NUM_WORKERS, pin_memory=True)
    return loaders


def class_weights_from(split):
    """Trọng số nghịch tần suất, chuẩn hoá để loss giữ nguyên thang đo."""
    counts = Counter(split[split["split"] == "train"]["class_id"])
    total = sum(counts.values())
    return torch.tensor([total / (len(CLASSES) * counts[i]) for i in range(len(CLASSES))],
                        dtype=torch.float, device=DEVICE)

## 2.3. Chỉ số đánh giá

Bài toán y tế nên bảng kết quả gồm cả nhóm chỉ số của học máy lẫn nhóm quen dùng
trong lâm sàng:

| Chỉ số | Ý nghĩa | Lưu ý khi đọc |
|---|---|---|
| `accuracy` | Tỉ lệ đúng chung | Đoán bừa PNEUMONIA đã được 62,5% trên tập test này |
| `precision` | Trong số ca bị gắn nhãn bệnh, bao nhiêu đúng | **Phụ thuộc tỉ lệ mắc bệnh** của tập đang đo |
| `recall` = độ nhạy | Trong số ca bệnh thật, bắt được bao nhiêu | Bỏ sót viêm phổi là lỗi nguy hiểm nhất |
| `specificity` = độ đặc hiệu | Trong số ca lành thật, nhận đúng bao nhiêu | Phơi bày tỉ lệ báo động giả |
| `f1` | Trung bình điều hoà precision và recall | Chỉ nhìn lớp dương |
| `bal_acc` | Trung bình độ nhạy và độ đặc hiệu | Không thưởng cho mô hình đoán lệch một phía |
| `auc` | Chất lượng xếp hạng, không phụ thuộc ngưỡng | Cao mà accuracy thấp ⇒ vấn đề ở ngưỡng |
| `pr_auc` | Như trên nhưng nhìn qua precision | |

**Về precision trong bối cảnh y tế.** Precision (giá trị tiên đoán dương) thay đổi
theo tỉ lệ mắc bệnh của quần thể: cùng một mô hình đem sang nơi có tỉ lệ mắc khác
sẽ cho precision khác, dù mô hình không đổi. Tập test ở đây có **62,5% ca bệnh** —
cao hơn thực tế phòng khám rất nhiều, nên precision đo được ở đây không chuyển
sang thực tế được.

Độ nhạy và độ đặc hiệu là thuộc tính của chính mô hình, không phụ thuộc tỉ lệ mắc.
Vì vậy chúng là số chính của báo cáo này, còn precision được ghi kèm chú thích.

Về việc chọn ưu tiên: với một công cụ **sàng lọc**, bỏ sót ca bệnh (FN) nguy hiểm
hơn báo động giả (FP), nên độ nhạy thường được ưu tiên. Precision chỉ trở thành
ràng buộc chính khi mỗi ca dương tính kéo theo can thiệp tốn kém hoặc xâm lấn.

## 2.4. Mô hình và huấn luyện

ResNet18 pretrained ImageNet, thay lớp cuối bằng đầu ra 2 lớp, fine-tune toàn bộ.
Loss là cross-entropy có trọng số lớp. Adam, learning rate giảm theo
`ReduceLROnPlateau` bám val F1, dừng sớm sau 5 epoch không cải thiện.

Checkpoint tốt nhất theo val F1 được **khôi phục trước khi chấm test**. Nếu không,
số liệu test sẽ thuộc về epoch mà vòng lặp tình cờ dừng lại chứ không phải mô hình
đã được chọn.

In [ ]:
def build_resnet18():
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
    return model.to(DEVICE)


@torch.no_grad()
def predict(model, loader):
    """(nhãn thật, xác suất PNEUMONIA) trên toàn bộ loader."""
    model.eval()
    labels_all, probs_all = [], []
    for images, labels in loader:
        logits = model(images.to(DEVICE, non_blocking=True))
        probs_all += torch.softmax(logits.float(), dim=1)[:, 1].cpu().tolist()
        labels_all += labels.tolist()
    return np.array(labels_all), np.array(probs_all)


def metrics_at(labels, probs, threshold=0.5):
    """Chấm điểm tại một ngưỡng. threshold=0.5 chính là argmax trên hai logit."""
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    sensitivity, specificity = tp / max(tp + fn, 1), tn / max(tn + fp, 1)
    return {
        "threshold": round(float(threshold), 4),
        "accuracy": float((labels == preds).mean()),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "specificity": float(specificity),
        "f1": f1_score(labels, preds, zero_division=0),
        "bal_acc": float((sensitivity + specificity) / 2),
        "auc": roc_auc_score(labels, probs),
        "pr_auc": average_precision_score(labels, probs),
        "confusion_matrix": confusion_matrix(labels, preds).tolist(),
    }


def tune_threshold(labels, probs):
    """Ngưỡng tối đa hoá balanced accuracy. Chỉ được gọi trên validation."""
    candidates = np.unique(np.clip(probs, 0.001, 0.999))
    if len(candidates) > 400:
        candidates = np.quantile(candidates, np.linspace(0, 1, 400))

    scores = []
    for threshold in candidates:
        preds = (probs >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
        scores.append((tp / max(tp + fn, 1) + tn / max(tn + fp, 1)) / 2)
    scores = np.array(scores)

    # Nhiều ngưỡng thường hoà nhau ở đỉnh. Lấy trung vị của vùng hoà thay vì điểm
    # đầu tiên, vì điểm đầu tiên luôn là ngưỡng thấp nhất trong vùng đó.
    tied = candidates[scores >= scores.max() - 1e-9]
    return float(np.median(tied)), float(scores.max())


def run_fold(fold_index, epochs=EPOCHS):
    log(f"\n{'=' * 62}\nfold {fold_index}\n{'=' * 62}")
    set_seed(SEED + fold_index)

    split = FOLDS[fold_index]
    loaders = make_loaders(split)
    model = build_resnet18()

    weights = class_weights_from(split) if CLASS_WEIGHTS else None
    if weights is not None:
        log("trọng số lớp:", [round(w, 3) for w in weights.tolist()])
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.3, patience=2)
    scaler = torch.amp.GradScaler("cuda")

    best_f1, best_epoch, best_state, stale = -1.0, 0, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for images, labels in loaders["train"]:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                loss = criterion(model(images), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * images.size(0)

        train_loss = running_loss / len(loaders["train"].dataset)
        val_labels, val_probs = predict(model, loaders["val"])
        val_metrics = metrics_at(val_labels, val_probs, 0.5)
        scheduler.step(val_metrics["f1"])

        marker = ""
        if val_metrics["f1"] > best_f1:
            best_f1, best_epoch, stale = val_metrics["f1"], epoch, 0
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
            marker = "  <- best"
        else:
            stale += 1

        log(f"epoch {epoch:>2}/{epochs}  loss {train_loss:.4f}  "
            f"val_f1 {val_metrics['f1']:.4f}  val_auc {val_metrics['auc']:.4f}{marker}")

        if stale >= PATIENCE:
            log(f"dừng sớm ở epoch {epoch}")
            break

    model.load_state_dict(best_state)
    log(f"khôi phục checkpoint tốt nhất (epoch {best_epoch}, val_f1 {best_f1:.4f})")

    val_labels, val_probs = predict(model, loaders["val"])
    test_labels, test_probs = predict(model, loaders["test"])
    threshold, val_score = tune_threshold(val_labels, val_probs)
    log(f"ngưỡng dò trên val: {threshold:.3f} (balanced accuracy {val_score:.4f})")

    torch.save(best_state, f"{WORK_DIR}/resnet18_fold{fold_index}.pth")
    split.to_csv(f"{WORK_DIR}/manifest_fold{fold_index}.csv", index=False)

    return {
        "fold": fold_index, "model": model, "loaders": loaders,
        "best_epoch": best_epoch, "threshold": threshold,
        "val_labels": val_labels, "val_probs": val_probs,
        "test_labels": test_labels, "test_probs": test_probs,
        "val": metrics_at(val_labels, val_probs, 0.5),
        "test": metrics_at(test_labels, test_probs, 0.5),
        "val_tuned": metrics_at(val_labels, val_probs, threshold),
        "test_tuned": metrics_at(test_labels, test_probs, threshold),
    }

---

# 3. Kết quả

## 3.1. Huấn luyện

In [ ]:
_t0 = time.time()
RUNS = [run_fold(i) for i in range(len(FOLDS))]
log(f"\ntổng thời gian: {(time.time() - _t0) / 60:.1f} phút")

In [ ]:
display(pd.DataFrame([
    {"fold": run["fold"], "epoch tốt nhất": run["best_epoch"],
     "val F1": round(run["val"]["f1"], 4), "val AUC": round(run["val"]["auc"], 4),
     "ngưỡng dò được": round(run["threshold"], 3)}
    for run in RUNS
]).set_index("fold"))

## 3.2. Kết quả trên tập test

Tập test giống nhau ở mọi fold, nên độ lệch chuẩn dưới đây phản ánh dao động do
khởi tạo và do cách chia validation, chứ không phải do tập đánh giá thay đổi.

In [ ]:
METRIC_COLS = ["accuracy", "precision", "recall", "specificity",
               "f1", "bal_acc", "auc", "pr_auc"]

per_fold = pd.DataFrame([
    {"fold": run["fold"], "split": split, "ngưỡng": mode,
     **{k: run[key][k] for k in METRIC_COLS}}
    for run in RUNS
    for split in ("val", "test")
    for mode, key in (("mặc định 0.5", split), ("đã dò", f"{split}_tuned"))
])
per_fold.to_csv(f"{WORK_DIR}/results_per_fold.csv", index=False)


def summarise(frame, columns=METRIC_COLS):
    grouped = frame.groupby(["split", "ngưỡng"])[list(columns)]
    mean, std = grouped.mean().round(4), grouped.std().round(4).fillna(0)
    return mean if len(RUNS) == 1 else mean.astype(str) + " ± " + std.astype(str)


print(f"Trung bình trên {len(RUNS)} fold (mean ± std)\n")
display(summarise(per_fold))

In [ ]:
def show_confusion(title, matrix):
    (tn, fp), (fn, tp) = matrix
    sensitivity, specificity = tp / max(tp + fn, 1), tn / max(tn + fp, 1)
    precision = tp / max(tp + fp, 1)
    print(f"\n{title}  (n={tn + fp + fn + tp})")
    print(f"  TN {tn:>4}   FP {fp:>4}")
    print(f"  FN {fn:>4}   TP {tp:>4}")
    print(f"  bỏ sót {fn}/{fn + tp} ca viêm phổi  → độ nhạy {sensitivity:.1%}")
    print(f"  báo nhầm {fp}/{tn + fp} ca bình thường → độ đặc hiệu {specificity:.1%}")
    print(f"  precision {precision:.1%}  |  balanced accuracy "
          f"{(sensitivity + specificity) / 2:.1%}")


# Trung bình xác suất của các fold: thường tốt hơn mọi fold đơn lẻ, không tốn
# thêm chi phí huấn luyện.
test_labels = RUNS[0]["test_labels"]
ens_probs = np.mean([run["test_probs"] for run in RUNS], axis=0)
ens_threshold = float(np.median([run["threshold"] for run in RUNS]))
ENSEMBLE = {"mặc định 0.5": metrics_at(test_labels, ens_probs, 0.5),
            "đã dò":        metrics_at(test_labels, ens_probs, ens_threshold)}

if len(RUNS) > 1:
    print("Ensemble — trung bình xác suất của các fold, chấm trên test\n")
    display(pd.DataFrame([
        {"ngưỡng": mode, **{k: round(entry[k], 4) for k in METRIC_COLS}}
        for mode, entry in ENSEMBLE.items()
    ]).set_index("ngưỡng"))

label = "ensemble" if len(RUNS) > 1 else "fold 0"
show_confusion(f"TEST @ ngưỡng mặc định 0.5 ({label})",
               ENSEMBLE["mặc định 0.5"]["confusion_matrix"])
show_confusion(f"TEST @ ngưỡng {ens_threshold:.3f} đã dò ({label})",
               ENSEMBLE["đã dò"]["confusion_matrix"])

## 3.3. Điểm làm việc

`argmax` trên hai logit tương đương ngưỡng cứng **0,5**. Ngưỡng đó hợp với tỉ lệ
lớp lúc huấn luyện (2,89:1) nhưng tập test có tỉ lệ khác (1,67:1), nên nó không
nhất thiết là điểm làm việc tốt nhất.

Ngưỡng được dò trên **validation**, không bao giờ trên test. Vì validation lấy từ
cùng phân bố với train còn test thì khác, ngưỡng dò được là điểm khởi đầu có cơ sở
chứ không phải bảo đảm — nên báo cáo trình bày cả hai.

Độ tản mát giữa các fold cho biết ngưỡng ổn định tới đâu.

In [ ]:
thresholds = [run["threshold"] for run in RUNS]
print("ngưỡng theo fold:", [round(t, 3) for t in thresholds])
if len(thresholds) > 1:
    print(f"trung vị {np.median(thresholds):.3f}  |  "
          f"khoảng [{min(thresholds):.3f}, {max(thresholds):.3f}]  |  "
          f"độ lệch chuẩn {np.std(thresholds):.3f}")

val_f1  = per_fold[(per_fold.split == "val")
                   & (per_fold["ngưỡng"] == "mặc định 0.5")]["f1"].mean()
test_f1 = per_fold[(per_fold.split == "test")
                   & (per_fold["ngưỡng"] == "mặc định 0.5")]["f1"].mean()
print(f"\nF1 validation {val_f1:.4f}  →  F1 test {test_f1:.4f}  "
      f"(chênh {val_f1 - test_f1:+.4f})")
print("Validation đã được dùng để chọn checkpoint nên số của nó lạc quan hơn;")
print("ngoài ra hai tập còn khác tỉ lệ lớp. Test mới là ước lượng khái quát hoá.")

---

# 4. Diễn giải mô hình

Chỉ số cao chưa đủ để tin một mô hình y tế. Grad-CAM cho thấy vùng ảnh mô hình dựa
vào; nếu vùng đó không phải nhu mô phổi thì kết quả tốt là do trùng hợp chứ không
do học được dấu hiệu bệnh.

Mô hình dùng ở đây là fold 0, đọc dự đoán ở đúng ngưỡng đã báo cáo ở phần 3 để
hình minh hoạ khớp với confusion matrix.

In [ ]:
class GradCAM:
    """Grad-CAM bằng hook trên một lớp tích chập.

    Mỗi kênh đặc trưng được đánh trọng số bằng gradient trung bình của nó với
    logit mục tiêu, cộng lại rồi lấy phần dương.
    """

    def __init__(self, model, target_layer):
        self.model, self.activations, self.gradients = model, None, None
        self._handles = [
            target_layer.register_forward_hook(self._save_activations),
            target_layer.register_full_backward_hook(self._save_gradients),
        ]

    def _save_activations(self, module, inputs, output):
        self.activations = output.detach()

    def _save_gradients(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def __call__(self, images, class_indices=None):
        self.model.zero_grad(set_to_none=True)
        logits = self.model(images)          # cần gradient nên không dùng no_grad
        if class_indices is None:
            class_indices = logits.argmax(dim=1)
        logits[torch.arange(len(images)), class_indices].sum().backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cams = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cams = F.interpolate(cams, size=images.shape[-2:],
                             mode="bilinear", align_corners=False)[:, 0]
        cams = cams - cams.amin(dim=(1, 2), keepdim=True)
        cams = cams / (cams.amax(dim=(1, 2), keepdim=True) + 1e-8)
        return cams.cpu().numpy(), logits.detach()

    def close(self):
        for handle in self._handles:
            handle.remove()


def denormalize(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor.cpu() * std + mean).clamp(0, 1).mean(0).numpy()


def model_input(row):
    """Lấy từ cache chứ không đọc lại file: eval_tf không còn bước resize."""
    return eval_tf(Image.fromarray(
        IMAGE_CACHE[int(row["cache_index"])])).unsqueeze(0).to(DEVICE)


XAI_RUN = RUNS[0]
xai_model = XAI_RUN["model"].eval()
# Chỉ đích danh lớp cuối của ResNet18 thay vì dò lớp tích chập cuối cùng — cách
# dò sẽ sai âm thầm nếu đổi kiến trúc.
cam_engine = GradCAM(xai_model, xai_model.layer4[-1])
THRESHOLD = XAI_RUN["threshold"]
print(f"fold {XAI_RUN['fold']} | layer4[-1] | ngưỡng {THRESHOLD:.3f}")

## 4.1. Chọn ca đại diện

Lấy một ca cho mỗi ô confusion matrix, chọn ca mô hình tự tin nhất — ca tự tin mà
vẫn sai là ca đáng mổ xẻ nhất. Lấy mẫu theo từng ô thay vì lấy đều theo chỉ số, vì
danh sách ảnh sắp theo lớp nên lấy đều sẽ lệch hẳn về PNEUMONIA.

In [ ]:
test_rows = FOLDS[XAI_RUN["fold"]]
test_rows = test_rows[test_rows["split"] == "test"].reset_index(drop=True)
test_rows = test_rows.assign(
    true=XAI_RUN["test_labels"],
    p_pneumonia=XAI_RUN["test_probs"],
    pred=(XAI_RUN["test_probs"] >= THRESHOLD).astype(int),
)

CM_CELLS = {
    "TN — NORMAL đúng":      (0, 0),
    "FP — báo động giả":     (0, 1),
    "FN — BỎ SÓT viêm phổi": (1, 0),
    "TP — PNEUMONIA đúng":   (1, 1),
}

picks = {}
for name, (true_label, pred_label) in CM_CELLS.items():
    subset = test_rows[(test_rows["true"] == true_label)
                       & (test_rows["pred"] == pred_label)]
    if subset.empty:
        print(f"{name:<24}: không có ca nào")
        continue
    confidence = subset["p_pneumonia"] if pred_label == 1 else 1 - subset["p_pneumonia"]
    picks[name] = subset.loc[confidence.idxmax()]
    print(f"{name:<24}: {len(subset):>3} ca  |  chọn {picks[name]['filename']}")

## 4.2. Bản đồ nhiệt

In [ ]:
figure, axes = plt.subplots(2, len(picks), figsize=(4.0 * len(picks), 8.6),
                            constrained_layout=True)
axes = np.atleast_2d(axes)

for column, (name, row) in enumerate(picks.items()):
    tensor = model_input(row)
    cams, logits = cam_engine(tensor)
    probability = torch.softmax(logits.float(), 1)[0, int(row["pred"])].item()
    grayscale = denormalize(tensor[0])

    axes[0, column].imshow(grayscale, cmap="gray")
    axes[0, column].set_title(f"{name}\nthật: {CLASSES[row['true']]}",
                              fontsize=11, pad=10)
    axes[1, column].imshow(grayscale, cmap="gray")
    axes[1, column].imshow(cams[0], cmap="jet", alpha=0.45)
    axes[1, column].text(
        0.03, 0.97, f"đoán: {CLASSES[row['pred']]} ({probability:.1%})",
        transform=axes[1, column].transAxes, va="top", ha="left",
        fontsize=10, fontweight="600", color="white",
        bbox=dict(boxstyle="round,pad=0.35", facecolor="#0b0b0b",
                  alpha=0.72, edgecolor="none"))
    for axis in axes[:, column]:
        axis.axis("off")

figure.suptitle("Grad-CAM — vùng ảnh mô hình dựa vào để quyết định", fontsize=13)
plt.show()

## 4.3. Kiểm tra định lượng

Quan sát vài tấm bằng mắt dễ dẫn tới kết luận cảm tính. Phép đo dưới đây tính tỉ lệ
khối lượng nhiệt rơi vào **viền ngoài 15%** của ảnh — vùng không thể chứa nhu mô
phổi — trên một mẫu ngẫu nhiên của tập test.

Mốc so sánh là chính tỉ lệ diện tích của viền đó: nếu nhiệt phân bố hoàn toàn ngẫu
nhiên, hai con số bằng nhau. Cao hơn mốc nghĩa là mô hình đang bám đặc trưng ngoài
phổi.

In [ ]:
def border_mass_fraction(cam, border=BORDER_FRAC):
    height, width = cam.shape
    margin_y, margin_x = int(height * border), int(width * border)
    interior = cam[margin_y:height - margin_y, margin_x:width - margin_x].sum()
    return float(1.0 - interior / (cam.sum() + 1e-8))


sample = test_rows.sample(n=min(120, len(test_rows)), random_state=SEED)
fractions = np.array([border_mass_fraction(cam_engine(model_input(row))[0][0])
                      for _, row in sample.iterrows()])
baseline = 1 - (1 - 2 * BORDER_FRAC) ** 2

print(f"n = {len(fractions)} ảnh test  |  viền ngoài {BORDER_FRAC:.0%}")
print(f"  trung vị       : {np.median(fractions):.1%}")
print(f"  trung bình     : {fractions.mean():.1%}")
print(f"  cao nhất       : {fractions.max():.1%}")
print(f"  mốc ngẫu nhiên : {baseline:.1%}")
print(f"  số ca vượt mốc : {(fractions > baseline).sum()}/{len(fractions)}")

cam_engine.close()

---

# 5. Kết luận và hạn chế

## Đã làm được

- Pipeline chạy đầu-cuối trên Kaggle, xử lý được các bất thường của bộ dữ liệu:
  hai kiểu đường dẫn mount, cây thư mục lồng trùng, file sidecar `._*`.
- Kiểm tra chất lượng dữ liệu có số liệu và tái chạy được.
- Chia dữ liệu theo bệnh nhân, xác nhận không bệnh nhân nào nằm ở hai split.
- Cross-validation nhiều fold, báo cáo khoảng dao động thay vì một con số đơn lẻ.
- Khôi phục checkpoint tốt nhất trước khi chấm test.
- Báo cáo cả ngưỡng mặc định lẫn ngưỡng dò trên validation.
- Kiểm tra diễn giải bằng Grad-CAM kèm phép đo định lượng.

## Hạn chế

- **Một bộ siêu tham số duy nhất.** Chưa dò learning rate, augmentation hay kích
  thước ảnh; kết quả chưa phải giới hạn của kiến trúc.
- **Một kiến trúc duy nhất.** Chưa so với DenseNet121, EfficientNet-B0 hay mô hình
  huấn luyện từ đầu.
- **Chưa bật chế độ tất định đầy đủ trên GPU**, nên chưa tái lập bit-for-bit.
- **Precision đo trên tập test có 62,5% ca bệnh** — cao hơn thực tế phòng khám,
  nên con số này không chuyển sang môi trường triển khai được.
- **Dữ liệu từ một trung tâm, chỉ gồm bệnh nhi 1–5 tuổi.** Không có cơ sở suy rộng
  sang người lớn hay cơ sở y tế khác nếu chưa kiểm chứng trên dữ liệu ngoài.
- **Nhãn nhị phân** không mô tả các bất thường khác có thể có trong nhóm bình thường.

## Hướng tiếp theo

1. Kiểm chứng trên một bộ dữ liệu ngoài trước khi nói tới khả năng khái quát hoá.
2. So sánh với DenseNet121 và EfficientNet-B0 dưới cùng giao thức đánh giá.
3. Nếu bản đồ nhiệt cho thấy mô hình bám vùng ngoài phổi: cắt hoặc che vùng phổi
   trước khi đưa vào mạng, rồi đo lại.
4. Chọn điểm làm việc theo yêu cầu lâm sàng — ưu tiên độ nhạy nếu dùng để sàng
   lọc — thay vì mặc định 0,5.

## Tệp xuất ra

| Tệp | Nội dung |
|---|---|
| `train_log.txt` | Nhật ký huấn luyện đầy đủ |
| `results_per_fold.csv` | Chỉ số của từng fold |
| `manifest_fold*.csv` | Danh sách ảnh chính xác của từng fold |
| `resnet18_fold*.pth` | Trọng số tốt nhất của từng fold |

Đính kèm `manifest_fold*.csv` khi nộp để người đọc truy ngược được đúng danh sách
ảnh đứng sau mỗi con số.